<a href="https://colab.research.google.com/github/nicopicomoco-sekihan-80/Antencoder-AntVLA/blob/main/V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1
# Qwen3-0.6B / GRPO / Colab
# ============================================================

!pip -q install -U \
    "transformers>=5.0.0" \
    "datasets>=4.0.0" \
    "accelerate" \
    "peft" \
    "bitsandbytes" \
    "trl"

import os
import re
import torch
from datasets import load_dataset

MODEL = "Qwen/Qwen3-0.6B"
OUT = "/content/qwen3-fast-thinking"

os.makedirs(OUT, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
      "GB")

# 数学データ
ds = load_dataset(
    "openai/gsm8k",
    "main",
    split="train"
)

# Colab T4でまず小規模実験
ds = ds.shuffle(seed=42).select(range(2000))

def extract_answer(x):
    m = re.search(r"####\s*(-?\d+(?:\.\d+)?)", x)
    return m.group(1) if m else ""

def extract_reasoning(x):
    # GSM8Kの長い解説をそのまま強制的に短くしない。
    # RLで「正解を維持しつつ短くする」ための初期データ。
    x = x.replace("####", "Final answer:")
    return x

def make_prompt(example):
    answer = extract_answer(example["answer"])

    return {
        "prompt": [
            {
                "role": "user",
                "content": example["question"]
            }
        ],
        "solution": answer
    }

ds = ds.map(make_prompt)

print(ds)
print(ds[0])

GPU: Tesla T4
VRAM: 14.6 GB


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer', 'prompt', 'solution'],
    num_rows: 2000
})
{'question': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?', 'answer': 'Mimi has 2 x 12 = <<2*12=24>>24 sea shells.\nKyle has 24 x 2 = <<24*2=48>>48 sea shells.\nLeigh has 48 / 3 = <<48/3=16>>16 sea shells.\n#### 16', 'prompt': [{'role': 'user', 'content': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?'}], 'solution': '16'}


In [ ]:
# ============================================================
# CELL 2
# Reward
# ============================================================

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    trust_remote_code=True
)


def get_completion_text(completion):
    """
    TRLのcompletion形式から本文を取得
    """
    if isinstance(completion, list):
        if len(completion) > 0:
            if isinstance(completion[0], dict):
                return completion[0].get("content", "")
    return str(completion)


def normalize_number(x):
    x = str(x).strip()

    # 数字以外をある程度除去
    x = x.replace(",", "")
    x = x.replace("$", "")

    m = re.search(
        r"-?\d+(?:\.\d+)?",
        x
    )

    return m.group(0) if m else ""


def extract_final_answer(text):
    """
    <answer>...</answer>
    または最後の数字を取得
    """

    m = re.search(
        r"<answer>\s*(.*?)\s*</answer>",
        text,
        flags=re.DOTALL
    )

    if m:
        return normalize_number(m.group(1))

    # Qwenの通常出力にも対応
    nums = re.findall(
        r"-?\d+(?:\.\d+)?",
        text
    )

    return nums[-1] if nums else ""


def extract_thinking(text):

    m = re.search(
        r"<think>(.*?)</think>",
        text,
        flags=re.DOTALL
    )

    if m:
        return m.group(1)

    return ""


def accuracy_reward(
    completions,
    solution,
    **kwargs
):

    rewards = []

    for completion, target in zip(
        completions,
        solution
    ):

        text = get_completion_text(
            completion
        )

        predicted = extract_final_answer(
            text
        )

        target = normalize_number(
            target
        )

        if predicted == target:
            rewards.append(10.0)
        else:
            rewards.append(-10.0)

    return rewards


def length_reward(
    completions,
    **kwargs
):

    rewards = []

    for completion in completions:

        text = get_completion_text(
            completion
        )

        thinking = extract_thinking(
            text
        )

        n = len(
            tokenizer(
                thinking,
                add_special_tokens=False
            )["input_ids"]
        )

        # ここは「短ければ短いほど良い」
        # にはしない。
        #
        # 目標長を80 token程度に置き、
        # それを大幅に超える場合だけ徐々に減点。
        if n <= 80:
            reward = 0.0

        elif n <= 160:
            reward = -0.5

        elif n <= 256:
            reward = -1.0

        else:
            reward = -2.0

        rewards.append(reward)

    return rewards


def format_reward(
    completions,
    **kwargs
):

    rewards = []

    for completion in completions:

        text = get_completion_text(
            completion
        )

        has_think = (
            "<think>" in text
            and "</think>" in text
        )

        if has_think:
            rewards.append(0.5)
        else:
            rewards.append(-0.5)

    return rewards


print("Reward functions ready.")

Reward functions ready.


In [ ]:
# ============================================================
# CELL 3
# QLoRA + GRPO
# ============================================================

import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import LoraConfig

from trl import (
    GRPOConfig,
    GRPOTrainer
)


# ------------------------------------------------------------
# 4bit
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False


# ------------------------------------------------------------
# LoRA
# ------------------------------------------------------------

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


# ------------------------------------------------------------
# GRPO
# ------------------------------------------------------------

training_args = GRPOConfig(

    output_dir=OUT,

    # T4向け
    per_device_train_batch_size=1,

    gradient_accumulation_steps=4,

    learning_rate=5e-6,

    num_train_epochs=1,

    # 1 promptにつき複数回答を生成
    num_generations=4,

    # Thinkingを残す
    max_completion_length=384,

    # Qwen3 Thinking
    chat_template_kwargs={
        "enable_thinking": True
    },

    # 安定性
    temperature=0.7,

    beta=0.02,

    logging_steps=5,

    save_strategy="steps",
    save_steps=100,

    save_total_limit=2,

    bf16=True,

    gradient_checkpointing=True,

    report_to="none",

    # 速度を測るためcompletionを記録
    log_completions=True,
)


trainer = GRPOTrainer(

    model=model,

    args=training_args,

    train_dataset=ds,

    reward_funcs=[
        accuracy_reward,
        length_reward,
        format_reward,
    ],

    peft_config=peft_config,

    processing_class=tokenizer,
)


print("Starting GRPO...")

trainer.train()

trainer.save_model(OUT)

tokenizer.save_pretrained(OUT)

print()
print("DONE")
print("Adapter:", OUT)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Starting GRPO...


Step,Training Loss
5,0.004637


╭──────────────────────────────────────────────────── Step 5 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                ┃ Completion            ┃ accuracy_reward ┃ length_reward ┃ format_reward ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                  │ <think>               │           10.00 │          0.00 │         -0.50 │      0.53 │ │
│ │ Jenny is working on a │ Okay, let's see.      │                 │               │               │           │ │
│ │ school project. She   │ Jenny is working on a │                 │               │               │           │ │
│ │ spends 10 hours doing │ school project. She   │                 │               │               │           │ │
│ │ research for the      │ spends 10 hours doing │                 │               │               │           │ │
│ │ project and 2 hours   │ research and 2 hours  │                 │               │               │           │ │
│ │ writing a proposal.   │ writing a proposal.   │                 │               │               │           │ │
│ │ If she has 20 hours   │ If she has 20 hours   │                 │               │               │           │ │
│ │ total to work on the  │ total to work on the  │                 │               │               │           │ │
│ │ project, how many     │ project, how many     │                 │               │               │           │ │
│ │ hours does she have   │ hours does she have   │                 │               │               │           │ │
│ │ left to write her     │ left to write her     │                 │               │               │           │ │
│ │ report?               │ report?               │                 │               │               │           │ │
│ │ assistant             │                       │                 │               │               │           │ │
│ │                       │ Hmm, first, I need to │                 │               │               │           │ │
│ │                       │ figure out the total  │                 │               │               │           │ │
│ │                       │ time she's already    │                 │               │               │           │ │
│ │                       │ spent on the project. │                 │               │               │           │ │
│ │                       │ She spent 10 hours on │                 │               │               │           │ │
│ │                       │ research and 2 hours  │                 │               │               │           │ │
│ │                       │ on writing a          │                 │               │               │           │ │
│ │                       │ proposal. So, adding  │                 │               │               │           │ │
│ │                       │ those together: 10    │                 │               │               │           │ │
│ │                       │ plus 2. Let me do     │                 │               │               │           │ │
│ │                       │ that calculation. 10  │                 │               │               │           │ │
│ │                       │ + 2 is 12. So, she    │                 │               │               │           │ │
│ │                       │ has 12 hours already  │                 │               │               │           │ │
│ │                       │ done. The problem     │                 │               │               │           │ │
│ │                       │ says she has 20 hours │                 │               │               │           │ │
│ │                       │ total to work, so the │                 │               │               │           │ │
│ │                       │ question is how many  │     

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 1
# Install + dataset
# Qwen3-0.6B / QLoRA / GRPO
# ============================================================

!pip -q install -U \
    "transformers>=5.0.0" \
    "datasets>=4.0.0" \
    "accelerate>=1.0.0" \
    "peft>=0.15.0" \
    "bitsandbytes>=0.45.0" \
    "trl>=0.20.0"

import os
import re
import time
import torch

from datasets import load_dataset
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen3-0.6B"
OUT = "/content/qwen3-0.6b-qlora-grpo"

os.makedirs(OUT, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        1
    ),
    "GB"
)

# ------------------------------------------------------------
# GSM8K
# ------------------------------------------------------------

dataset = load_dataset(
    "openai/gsm8k",
    "main",
    split="train"
)

# 今回は200問だけ
dataset = dataset.shuffle(seed=42).select(range(200))


def extract_answer(text):
    m = re.search(
        r"####\s*(-?\d+(?:\.\d+)?)",
        text
    )

    if m:
        return m.group(1)

    return ""


def make_example(x):

    return {
        "prompt": [
            {
                "role": "user",
                "content": x["question"]
            }
        ],

        "solution": extract_answer(
            x["answer"]
        )
    }


dataset = dataset.map(
    make_example,
    remove_columns=dataset.column_names
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    trust_remote_code=True
)

print(dataset)
print(dataset[0])

GPU: Tesla T4
VRAM: 14.6 GB


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'solution'],
    num_rows: 200
})
{'prompt': [{'role': 'user', 'content': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?'}], 'solution': '16'}


In [ ]:
# ============================================================
# CELL 2
# Reward functions
# ============================================================

def completion_text(completion):

    if isinstance(completion, list):

        if len(completion) > 0:

            if isinstance(completion[0], dict):
                return completion[0].get(
                    "content",
                    ""
                )

    return str(completion)


def normalize_number(text):

    text = str(text)

    # カンマ除去
    text = text.replace(",", "")

    # $などを除去
    text = text.replace("$", "")

    nums = re.findall(
        r"-?\d+(?:\.\d+)?",
        text
    )

    if not nums:
        return ""

    return nums[-1]


def extract_final_answer(text):

    # <answer>...</answer>
    m = re.search(
        r"<answer>\s*(.*?)\s*</answer>",
        text,
        re.DOTALL
    )

    if m:
        return normalize_number(
            m.group(1)
        )

    # Answer: 8
    m = re.search(
        r"(?:answer|final answer)\s*[:：]\s*(-?\d+(?:\.\d+)?)",
        text,
        re.IGNORECASE
    )

    if m:
        return normalize_number(
            m.group(1)
        )

    return normalize_number(text)


def extract_thinking(text):

    m = re.search(
        r"<think>(.*?)</think>",
        text,
        re.DOTALL
    )

    if m:
        return m.group(1)

    # Qwen3がタグなしで出した場合
    return text


# ------------------------------------------------------------
# 1. Accuracy
# ------------------------------------------------------------

def accuracy_reward(
    completions,
    solution,
    **kwargs
):

    rewards = []

    for completion, target in zip(
        completions,
        solution
    ):

        text = completion_text(
            completion
        )

        predicted = extract_final_answer(
            text
        )

        target = normalize_number(
            target
        )

        if predicted == target:

            # 最重要
            rewards.append(10.0)

        else:

            rewards.append(-10.0)

    return rewards


# ------------------------------------------------------------
# 2. Thinking length
# ------------------------------------------------------------

def thinking_length_reward(
    completions,
    **kwargs
):

    rewards = []

    for completion in completions:

        text = completion_text(
            completion
        )

        thinking = extract_thinking(
            text
        )

        n = len(
            tokenizer(
                thinking,
                add_special_tokens=False
            )["input_ids"]
        )

        # ----------------------------------------------------
        # 短ければ無条件に高得点、にはしない
        #
        # 0～40   : 0
        # 40～80  : -0.2
        # 80～120 : -0.5
        # 120～180: -1
        # 180～256: -2
        # >256    : -3
        # ----------------------------------------------------

        if n <= 40:
            r = 0.0

        elif n <= 80:
            r = -0.2

        elif n <= 120:
            r = -0.5

        elif n <= 180:
            r = -1.0

        elif n <= 256:
            r = -2.0

        else:
            r = -3.0

        rewards.append(r)

    return rewards


# ------------------------------------------------------------
# 3. Repetition penalty
# ------------------------------------------------------------

def repetition_reward(
    completions,
    **kwargs
):

    rewards = []

    for completion in completions:

        text = completion_text(
            completion
        )

        # よくある冗長表現
        patterns = [
            "wait",
            "let me check",
            "let me make sure",
            "double-check",
            "again",
            "i think that's",
            "yep, that seems",
        ]

        count = 0

        lower = text.lower()

        for p in patterns:
            count += lower.count(p)

        # 最大 -1
        reward = -min(
            count * 0.1,
            1.0
        )

        rewards.append(reward)

    return rewards


# ------------------------------------------------------------
# 4. Format
# ------------------------------------------------------------

def format_reward(
    completions,
    **kwargs
):

    rewards = []

    for completion in completions:

        text = completion_text(
            completion
        )

        if "</think>" in text:

            rewards.append(0.5)

        else:

            rewards.append(0.0)

    return rewards


print("Rewards ready.")

Rewards ready.


In [ ]:
# Qwen3-0.6B / QLoRA + GRPO / 1-cell Colab
# 目的：正答率を維持しながら、不要なCoT・自己反復を削減

!pip -q install -U "transformers>=5.0.0" "datasets>=4.0.0" accelerate peft bitsandbytes "trl>=0.20.0"

import os, re, time, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import GRPOConfig, GRPOTrainer

MODEL = "Qwen/Qwen3-0.6B"
OUT = "/content/qwen3-0.6b-qlora-grpo"

os.makedirs(OUT, exist_ok=True)

print("=" * 60)
print("Qwen3-0.6B QLoRA + GRPO")
print("=" * 60)
print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    trust_remote_code=True
)

# -----------------------------
# Dataset
# -----------------------------

raw = load_dataset(
    "openai/gsm8k",
    "main",
    split="train"
).shuffle(seed=42).select(range(200))

def make_example(x):
    m = re.search(
        r"####\s*(-?\d+(?:\.\d+)?)",
        x["answer"]
    )

    return {
        "prompt": [
            {
                "role": "user",
                "content": x["question"]
            }
        ],
        "solution": m.group(1) if m else ""
    }

dataset = raw.map(
    make_example,
    remove_columns=raw.column_names
)

# -----------------------------
# Utility
# -----------------------------

def get_text(c):
    if isinstance(c, list) and len(c) > 0:
        if isinstance(c[0], dict):
            return c[0].get("content", "")
    return str(c)

def normalize_number(x):
    x = str(x)
    x = x.replace(",", "")
    x = x.replace("$", "")

    nums = re.findall(
        r"-?\d+(?:\.\d+)?",
        x
    )

    return nums[-1] if nums else ""

def extract_answer(text):

    m = re.search(
        r"<answer>\s*(.*?)\s*</answer>",
        text,
        re.DOTALL | re.IGNORECASE
    )

    if m:
        return normalize_number(m.group(1))

    m = re.search(
        r"(?:answer|final answer)\s*[:：]\s*(-?\d+(?:\.\d+)?)",
        text,
        re.IGNORECASE
    )

    if m:
        return normalize_number(m.group(1))

    return normalize_number(text)

def extract_think(text):

    m = re.search(
        r"<think>(.*?)</think>",
        text,
        re.DOTALL | re.IGNORECASE
    )

    if m:
        return m.group(1)

    return text

# -----------------------------
# Rewards
# -----------------------------

def accuracy_reward(
    completions,
    solution,
    **kwargs
):

    rewards = []

    for completion, target in zip(
        completions,
        solution
    ):

        text = get_text(completion)

        pred = extract_answer(text)
        target = normalize_number(target)

        if pred == target:
            rewards.append(10.0)
        else:
            rewards.append(-10.0)

    return rewards


def length_reward(
    completions,
    **kwargs
):

    rewards = []

    for completion in completions:

        text = get_text(completion)
        think = extract_think(text)

        n = len(
            tokenizer(
                think,
                add_special_tokens=False
            )["input_ids"]
        )

        # 正答率を優先し、
        # 必要以上のCoTだけ緩やかに減点

        if n <= 40:
            r = 0.0

        elif n <= 80:
            r = -0.2

        elif n <= 120:
            r = -0.5

        elif n <= 180:
            r = -1.0

        elif n <= 256:
            r = -2.0

        else:
            r = -3.0

        rewards.append(r)

    return rewards


def repetition_reward(
    completions,
    **kwargs
):

    patterns = [
        "wait",
        "let me check",
        "let me make sure",
        "double-check",
        "again",
        "i think that's",
        "yep, that seems"
    ]

    rewards = []

    for completion in completions:

        text = get_text(completion).lower()

        count = sum(
            text.count(p)
            for p in patterns
        )

        rewards.append(
            -min(
                count * 0.1,
                1.0
            )
        )

    return rewards


def format_reward(
    completions,
    **kwargs
):

    return [
        0.5
        if "</think>" in get_text(c)
        else 0.0
        for c in completions
    ]

# -----------------------------
# QLoRA
# -----------------------------

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

# -----------------------------
# GRPO
# -----------------------------

args = GRPOConfig(
    output_dir=OUT,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,

    num_generations=4,

    max_completion_length=256,

    learning_rate=5e-6,

    num_train_epochs=1,

    beta=0.02,

    temperature=0.7,
    top_p=0.95,

    bf16=True,

    gradient_checkpointing=True,

    logging_steps=5,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    report_to="none",

    log_completions=True,

    chat_template_kwargs={
        "enable_thinking": True
    }
)

trainer = GRPOTrainer(
    model=model,
    args=args,
    train_dataset=dataset,

    reward_funcs=[
        accuracy_reward,
        length_reward,
        repetition_reward,
        format_reward
    ],

    peft_config=lora,
    processing_class=tokenizer
)

# -----------------------------
# Train
# -----------------------------

print()
print("Starting GRPO...")
print("Samples:", len(dataset))
print("Generations:", 4)
print("Max completion:", 256)
print()

start = time.time()

trainer.train()

elapsed = time.time() - start

print()
print("=" * 60)
print(
    "Training finished:",
    round(elapsed / 60, 1),
    "minutes"
)
print("=" * 60)

# -----------------------------
# Save
# -----------------------------

trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)

print("Saved to:", OUT)

# -----------------------------
# Quick inference
# -----------------------------

questions = [
    "17 × 24 を計算してください。",
    "125 + 378 を計算してください。",
    "Pythonでリストの最大値を返す関数を書いてください。"
]

print()
print("=" * 60)
print("Quick test")
print("=" * 60)

model.eval()

for q in questions:

    messages = [
        {
            "role": "user",
            "content": q
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            use_cache=True
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - t0

    generated = outputs[
        0,
        inputs.input_ids.shape[-1]:
    ]

    result = tokenizer.decode(
        generated,
        skip_special_tokens=False
    )

    think = extract_think(result)

    think_tokens = len(
        tokenizer(
            think,
            add_special_tokens=False
        )["input_ids"]
    )

    print()
    print("Q:", q)
    print(
        "time:",
        round(elapsed, 2),
        "sec",
        "| thinking tokens:",
        think_tokens
    )
    print(result)

print()
print("DONE")

Qwen3-0.6B QLoRA + GRPO
GPU: Tesla T4


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

ValueError: generation_batch_size (2) must be divisible by num_generations (4).

In [19]:
# ============================================================
# Qwen3-0.6B / QLoRA + GRPO
# 1セル・Google Colab版
#
# 目的:
#   - 数学の正答率を維持
#   - Coding/数学で必要な思考は最大1024 tokenまで許可
#   - 簡単な問題の無駄なCoTをRLで削減
#   - "Wait...", "Let me check..." などの自己反復を削減
# ============================================================

!pip -q install -U \
    "transformers>=5.0.0" \
    "datasets>=4.0.0" \
    accelerate \
    peft \
    bitsandbytes \
    "trl>=0.20.0"

import os
import re
import time
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    PeftModel
)

from trl import (
    GRPOConfig,
    GRPOTrainer
)


# ============================================================
# CONFIG
# ============================================================

MODEL = "Qwen/Qwen3-0.6B"

OUT = "/content/qwen3-0.6b-qlora-grpo"

NUM_SAMPLES = 50

NUM_GENERATIONS = 2

MAX_COMPLETION_LENGTH = 1024

os.makedirs(OUT, exist_ok=True)


# ============================================================
# GPU
# ============================================================

print("=" * 70)
print("Qwen3-0.6B QLoRA + GRPO")
print("=" * 70)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

gpu_mem = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)

print(
    "VRAM:",
    round(gpu_mem, 1),
    "GB"
)


# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    trust_remote_code=True
)


# ============================================================
# DATASET
# ============================================================

print()
print("Loading GSM8K...")

raw_dataset = load_dataset(
    "openai/gsm8k",
    "main",
    split="train"
)

raw_dataset = (
    raw_dataset
    .shuffle(seed=42)
    .select(range(NUM_SAMPLES))
)


def make_example(x):

    m = re.search(
        r"####\s*(-?\d+(?:\.\d+)?)",
        x["answer"]
    )

    solution = (
        m.group(1)
        if m
        else ""
    )

    return {
        "prompt": [
            {
                "role": "user",
                "content": x["question"]
            }
        ],

        "solution": solution
    }


dataset = raw_dataset.map(
    make_example,
    remove_columns=raw_dataset.column_names
)

print(
    "Dataset:",
    len(dataset),
    "examples"
)


# ============================================================
# TEXT UTILITIES
# ============================================================

def get_text(completion):

    if isinstance(completion, list):

        if len(completion) > 0:

            if isinstance(
                completion[0],
                dict
            ):

                return completion[0].get(
                    "content",
                    ""
                )

    return str(completion)


def normalize_number(text):

    text = str(text)

    text = text.replace(
        ",",
        ""
    )

    text = text.replace(
        "$",
        ""
    )

    numbers = re.findall(
        r"-?\d+(?:\.\d+)?",
        text
    )

    if not numbers:
        return ""

    return numbers[-1]


def extract_answer(text):

    # --------------------------------------------------------
    # <answer>...</answer>
    # --------------------------------------------------------

    m = re.search(
        r"<answer>\s*(.*?)\s*</answer>",
        text,
        re.DOTALL | re.IGNORECASE
    )

    if m:

        return normalize_number(
            m.group(1)
        )


    # --------------------------------------------------------
    # Answer: 123
    # --------------------------------------------------------

    m = re.search(
        r"(?:answer|final answer)"
        r"\s*[:：]\s*"
        r"(-?\d+(?:\.\d+)?)",
        text,
        re.IGNORECASE
    )

    if m:

        return normalize_number(
            m.group(1)
        )


    # --------------------------------------------------------
    # 最後の数字
    # --------------------------------------------------------

    return normalize_number(
        text
    )


def extract_thinking(text):

    m = re.search(
        r"<think>(.*?)</think>",
        text,
        re.DOTALL | re.IGNORECASE
    )

    if m:

        return m.group(1)

    return text


# ============================================================
# REWARD 1
# ACCURACY
#
# 最重要。
#
# 正解      +10
# 不正解    -10
# ============================================================

def accuracy_reward(
    completions,
    solution,
    **kwargs
):

    rewards = []

    for completion, target in zip(
        completions,
        solution
    ):

        text = get_text(
            completion
        )

        predicted = extract_answer(
            text
        )

        target = normalize_number(
            target
        )

        if predicted == target:

            rewards.append(
                10.0
            )

        else:

            rewards.append(
                -10.0
            )

    return rewards


# ============================================================
# REWARD 2
# THINKING LENGTH
#
# 短ければ無条件に高得点にはしない。
#
# 正解を維持しながら、
# 不必要に長いCoTだけを抑える。
# ============================================================

def length_reward(
    completions,
    **kwargs
):

    rewards = []

    for completion in completions:

        text = get_text(
            completion
        )

        think = extract_thinking(
            text
        )

        token_count = len(
            tokenizer(
                think,
                add_special_tokens=False
            )["input_ids"]
        )


        # ----------------------------------------------------
        # かなり短い
        # ----------------------------------------------------

        if token_count <= 32:

            reward = 0.0


        # ----------------------------------------------------
        # 短い
        # ----------------------------------------------------

        elif token_count <= 64:

            reward = -0.15


        # ----------------------------------------------------
        # 普通
        # ----------------------------------------------------

        elif token_count <= 128:

            reward = -0.4


        # ----------------------------------------------------
        # やや長い
        # ----------------------------------------------------

        elif token_count <= 256:

            reward = -1.0


        # ----------------------------------------------------
        # 長い
        # ----------------------------------------------------

        elif token_count <= 384:

            reward = -1.8


        # ----------------------------------------------------
        # かなり長い
        # ----------------------------------------------------

        elif token_count <= 512:

            reward = -2.5


        # ----------------------------------------------------
        # 512～1024
        # ----------------------------------------------------

        elif token_count <= 768:

            reward = -3.2


        # ----------------------------------------------------
        # 768～1024
        # ----------------------------------------------------

        else:

            reward = -4.0


        rewards.append(
            reward
        )

    return rewards


# ============================================================
# REWARD 3
# REPETITION
#
# Qwen3-0.6Bで出やすい
#
# "Wait..."
# "Let me check..."
# "Let me make sure..."
#
# などを抑制する。
# ============================================================

def repetition_reward(
    completions,
    **kwargs
):

    patterns = [

        "wait",

        "let me check",

        "let me make sure",

        "double-check",

        "double check",

        "again",

        "i think that's",

        "yep, that seems",

        "let me verify",

        "let me confirm",

    ]

    rewards = []

    for completion in completions:

        text = get_text(
            completion
        ).lower()

        count = 0

        for pattern in patterns:

            count += text.count(
                pattern
            )


        # 最大 -1.5
        penalty = min(
            count * 0.15,
            1.5
        )

        rewards.append(
            -penalty
        )

    return rewards


# ============================================================
# REWARD 4
# FORMAT
# ============================================================

def format_reward(
    completions,
    **kwargs
):

    rewards = []

    for completion in completions:

        text = get_text(
            completion
        )

        if "</think>" in text:

            rewards.append(
                0.5
            )

        else:

            rewards.append(
                0.0
            )

    return rewards


# ============================================================
# 4-BIT QUANTIZATION
# ============================================================

print()
print("Loading 4-bit model...")


bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.bfloat16,

    bnb_4bit_use_double_quant=True
)


# ============================================================
# MODEL
# ============================================================

model = AutoModelForCausalLM.from_pretrained(

    MODEL,

    quantization_config=bnb_config,

    device_map="auto",

    trust_remote_code=True
)

model.config.use_cache = False


# ============================================================
# LORA
# ============================================================

lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    target_modules=[

        "q_proj",

        "k_proj",

        "v_proj",

        "o_proj",

        "gate_proj",

        "up_proj",

        "down_proj",

    ]
)


# ============================================================
# GRPO CONFIG
#
# 重要:
#
# generation_batch_size=4
#
# num_generations=4
#
# なので、
#
# 4 % 4 == 0
#
# となり、今回のエラーを回避する。
# ============================================================

print()
print("Building GRPO configuration...")


grpo_args = GRPOConfig(

    output_dir=OUT,


    # --------------------------------------------------------
    # Training batch
    # --------------------------------------------------------

    per_device_train_batch_size=1,

    gradient_accumulation_steps=2,


    # --------------------------------------------------------
    # Generation batch
    #
    # TRL最新版では
    # generation_batch_size が
    # num_generations の倍数である必要がある。
    # --------------------------------------------------------

    generation_batch_size=NUM_GENERATIONS,


    # --------------------------------------------------------
    # GRPO
    # --------------------------------------------------------

    num_generations=NUM_GENERATIONS,


    # --------------------------------------------------------
    # 最大生成量
    #
    # 思考 + 回答
    #
    # 最大1024 token
    # --------------------------------------------------------

    max_completion_length=MAX_COMPLETION_LENGTH,


    # --------------------------------------------------------
    # Learning rate
    # --------------------------------------------------------

    learning_rate=5e-6,

    num_train_epochs=1,


    # --------------------------------------------------------
    # KL
    # --------------------------------------------------------

    beta=0.02,


    # --------------------------------------------------------
    # Generation
    # --------------------------------------------------------

    temperature=0.7,

    top_p=0.95,


    # --------------------------------------------------------
    # QLoRA / T4
    # --------------------------------------------------------

    bf16=True,

    gradient_checkpointing=True,


    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    logging_steps=5,

    logging_first_step=True,

    log_on_each_node=False,


    # --------------------------------------------------------
    # Checkpoints
    # --------------------------------------------------------

    save_strategy="steps",

    save_steps=50,

    save_total_limit=2,


    # --------------------------------------------------------
    # No W&B
    # --------------------------------------------------------

    report_to="none",


    # --------------------------------------------------------
    # Completion logging
    # --------------------------------------------------------

    log_completions=True,


    # --------------------------------------------------------
    # Qwen3 thinking
    # --------------------------------------------------------

    chat_template_kwargs={

        "enable_thinking": True

    }
)


# ============================================================
# TRAINER
# ============================================================

trainer = GRPOTrainer(

    model=model,

    args=grpo_args,

    train_dataset=dataset,

    reward_funcs=[

        accuracy_reward,

        length_reward,

        repetition_reward,

        format_reward,

    ],

    peft_config=lora_config,

    processing_class=tokenizer
)


# ============================================================
# TRAIN
# ============================================================

print()
print("=" * 70)
print("STARTING GRPO")
print("=" * 70)

print(
    "Samples:",
    NUM_SAMPLES
)

print(
    "Generations:",
    NUM_GENERATIONS
)

print(
    "Max completion:",
    MAX_COMPLETION_LENGTH
)

print()
print(
    "Goal: preserve accuracy while reducing unnecessary CoT."
)
print()

start_time = time.time()


trainer.train()


elapsed = (
    time.time()
    - start_time
)


# ============================================================
# SAVE
# ============================================================

print()
print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

print(
    "Training time:",
    round(
        elapsed / 60,
        1
    ),
    "minutes"
)


trainer.save_model(
    OUT
)

tokenizer.save_pretrained(
    OUT
)


print()
print(
    "LoRA adapter saved to:"
)

print(
    OUT
)


# ============================================================
# QUICK TEST
# ============================================================

print()
print("=" * 70)
print("QUICK TEST")
print("=" * 70)


test_questions = [

    "3+5*3=",

    "17 × 24 を計算してください。",

    "125 + 378 を計算してください。",

    "250の18%を計算してください。",

    "3x + 5 = 20 のxを求めてください。",

    "Pythonでリストの最大値を返す関数を書いてください。",

    "Pythonで文字列を逆順にする関数を書いてください。",

]


model.eval()


for question in test_questions:

    messages = [

        {
            "role": "user",

            "content": question
        }

    ]


    prompt = tokenizer.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True,

        enable_thinking=True
    )


    inputs = tokenizer(

        prompt,

        return_tensors="pt"

    ).to(model.device)


    if torch.cuda.is_available():

        torch.cuda.synchronize()


    t0 = time.perf_counter()


    with torch.no_grad():

        outputs = model.generate(

            **inputs,

            max_new_tokens=MAX_COMPLETION_LENGTH,

            do_sample=False,

            use_cache=True

        )


    if torch.cuda.is_available():

        torch.cuda.synchronize()


    elapsed_generation = (
        time.perf_counter()
        - t0
    )


    generated = outputs[

        0,

        inputs.input_ids.shape[-1]:

    ]


    result = tokenizer.decode(

        generated,

        skip_special_tokens=False

    )


    thinking_text = extract_thinking(
        result
    )


    thinking_tokens = len(

        tokenizer(

            thinking_text,

            add_special_tokens=False

        )["input_ids"]

    )


    output_tokens = len(
        generated
    )


    print()
    print("-" * 70)

    print(
        "Q:",
        question
    )

    print(
        "Time:",
        round(
            elapsed_generation,
            2
        ),
        "sec"
    )

    print(
        "Thinking tokens:",
        thinking_tokens
    )

    print(
        "Output tokens:",
        output_tokens
    )

    print()
    print(result)


print()
print("=" * 70)
print("DONE")
print("=" * 70)


Qwen3-0.6B QLoRA + GRPO
GPU: Tesla T4
VRAM: 14.6 GB

Loading GSM8K...


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Dataset: 50 examples

Loading 4-bit model...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]


Building GRPO configuration...


KeyboardInterrupt: 